# Author: Christina Blackwell
# Team A
## This notebook runs DeepSEM on the training data.
## Originally I ran DeepSEM on the entire training dataset all at once, thinking that DeepSEM would pull cell type information from the h5ad file, which has the multiome metadata included. After looking at the source code more closely I realized the dataset needed to be split into chunks for each cell type; looking at the demo dataset they used for the cell type inference task in the DeepSEM repository, you will notice the cell IDs all correspond to the same type of cell.
## DeepSEM GitHub: https://github.com/HantaoShu/DeepSEM
## The APIs for the [pandas](https://pandas.pydata.org/docs/reference/index.html), [AnnData](https://anndata.readthedocs.io/en/stable/api.html), and [NumPy](https://numpy.org/doc/stable/reference/index.html) libraries were referenced for this assignment.


# AI Statement
## During the preparation of this notebook, I used Google Gemini 3 to assist with debugging, determining the correct command line arguments to download the datasets, and assisting with identifying correct Python syntax. I subsequently reviewed, edited, and manually verified all logic for accuracy and security.¶

In [1]:
# Install dependencies
%pip install torch
%pip install numpy
%pip install pandas
%pip install scikit-learn
%pip install anndata
%pip install scanpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 25.2 MB/s eta 0:00:00


In [2]:
# Import libraries
import pandas as pd
import numpy as np
import subprocess
import os
import anndata as ad
from google.colab import files
from google.colab import drive
import shutil

## Download the DeepSEM model and datasets.

In [3]:
# Clone the repo structure WITHOUT downloading any file contents (--sparse)
!git clone -q --filter=blob:none --sparse https://github.com/cblackwe11/TCSS-478-588-Project.git temp_repo

# Navigate into the temporary repo and download the DeepSEM-master folder
!cd temp_repo && git sparse-checkout set DeepSEM-master

# Move the isolated folder out to the main Colab directory
if os.path.exists('temp_repo/DeepSEM-master'):

    # If a previous run left a folder behind, remove it first
    if os.path.exists('./DeepSEM-master'):
        shutil.rmtree('./DeepSEM-master')
    shutil.move('temp_repo/DeepSEM-master', './DeepSEM-master')

# Delete the temporary repo shell to keep things clean
shutil.rmtree('temp_repo')

# Download datasets
files_to_download = [ "CHOOSE-multiome-wt-log-norm.csv.gz", "CHOOSE-multiome-wt-metadata.csv"]
for file in files_to_download:
    url = f"https://github.com/cblackwe11/TCSS-478-588-Project/raw/main/{file}"
    !wget -q -O {file} {url}

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 6 (delta 0), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), 1.86 MiB | 5.12 MiB/s, done.
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 29 (delta 6), reused 29 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 27.27 MiB | 15.08 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Updating files: 100% (35/35), done.


## DeepSEM expects columns to correspond to genes and rows to correspond to cells per the README on the GitHub page, so the data frame containing expression values must be transposed before it can be used as input. Also, since we are building a cell type specific GRN, we need to pull the celltype_jf data from the metadata file.

## Note: The DeepSEM source code is pulled from our project repository and not the DeepSEM repository because a line needed to be added to the `Model.py` script to stop it from crashing when running the model with Google Colab. After line 25, I added `dropout_mask = dropout_mask.to(real.device)`, which is the fix suggested by Gemini.

In [4]:
# Read in the expression data
df_expression = pd.read_csv("CHOOSE-multiome-wt-log-norm.csv.gz", index_col=0)

# Transpose the data frame
df_transposed = df_expression.T

# Read in the metadata file
df_meta = pd.read_csv("CHOOSE-multiome-wt-metadata.csv", index_col=0)

# Align to the transposed expression matrix
df_meta_aligned = df_meta.loc[df_transposed.index]

# Get all of the possible cell types
cell_types = df_meta_aligned['celltype_jf'].unique()

print(df_meta_aligned.head())
print(df_transposed.head())

                      orig.ident  nCount_RNA  nFeature_RNA  nCount_ATAC  \
AAAGAACGTACAAGTA-1_1      137116        4191          2139            0   
AACACACGTACAACGG-1_1      137116        9199          4531            0   
AACTTCTAGAAGATCT-1_1      137116        6288          2694            0   
ACCAAACTCCGATTAG-1_1      137116       19876          5106            0   
ACGTAACAGAAAGCGA-1_1      137116        6646          2922            0   

                      nFeature_ATAC  barcode  linas  percent_mito  \
AAAGAACGTACAAGTA-1_1              0      NaN    NaN           NaN   
AACACACGTACAACGG-1_1              0      NaN    NaN           NaN   
AACTTCTAGAAGATCT-1_1              0      NaN    NaN           NaN   
ACCAAACTCCGATTAG-1_1              0      NaN    NaN           NaN   
ACGTAACAGAAAGCGA-1_1              0      NaN    NaN           NaN   

                      RNA_snn_res.0.8  seurat_clusters  ...      gRNA  \
AAAGAACGTACAAGTA-1_1              NaN               54  ...  

In [5]:
drive.mount('/content/drive')

Mounted at /content/drive


## Run DeepSEM with the transposed expression data. The dataset is split by each cell type so DeepSEM can perform the cell type specific inference task.
## The output is written to the drive so the `apply_viper_evaluation` notebook has access to the inferred GRN files. That notebook assumes the `DeepSEM_Results` folder resides within `/content/drive/MyDrive/` so moving it will cause things to break.

In [ ]:
os.chdir('/content/drive/MyDrive/')
output_dir = 'DeepSEM_Results/'
os.makedirs(output_dir, exist_ok=True)
os.chdir(output_dir)
print(os.getcwd())

for c_type in cell_types:
    print(f"Processing cell type: {c_type}")

    # Check if this cell type has already been processed
    if os.path.exists(f"{c_type}/GRN_inference_result.tsv"):
        print(f"Skipping {c_type}, already processed.")
        continue

    # Isolate the cell IDs for this specific type
    cells_in_type = df_meta_aligned[df_meta_aligned['celltype_jf'] == c_type].index
    df_subset = df_transposed.loc[cells_in_type]

    # Filter out low/zero variance genes from this subset to prevent NaN errors when the model divides by variance
    df_subset = df_subset.loc[:, df_subset.std(axis=0) > 1e-5]


    # Create an AnnData object, X = the expression matrix
    adata_subset = ad.AnnData(X=df_subset)

    # Create a temporary h5ad file to be used as input
    subset_file = f"temp_input_{c_type}.h5ad"
    adata_subset.write_h5ad(subset_file)

    # Build the DeepSEM command
    # The default # of epochs is 150 per the source code, changed to 120 to try to decrease runtime
    deepsem_command = [
        "python", "/content/DeepSEM-master/main.py",
        "--task", "celltype_GRN",
        "--data_file", subset_file,
        "--save_name", c_type,             # The path to save to
        "--setting", "test",               # Researchers' preconfigured hyperparameter settings
        "--n_epochs", "120"
    ]

    # Run DeepSEM
    try:
        result = subprocess.run(deepsem_command, capture_output=True, text=True, check=True)
        expected_file = f"{c_type}/GRN_inference_result.tsv"
        if os.path.exists(expected_file) and os.path.getsize(expected_file) > 0:
            print(f"{c_type} finished processing.")
        else:
            print(f"WARNING: Process did not crash, but no data was found.")
            # Print DeepSEM logs
            print(result.stdout)
        # Remove the temporary .h5ad file to save disk space
        os.remove(subset_file)
    except subprocess.CalledProcessError as e:
        print(f"Error processing {c_type}: {e}")
        # Display console output for debugging
        print(e.stdout)
        print(e.stderr)
        # Ensure cleanup still happens even if the run fails
        if os.path.exists(subset_file):
            os.remove(subset_file)

## The original method where the entire dataset is fed into DeepSEM.

In [ ]:
# Create the AnnData object, obs = the observations (metadata info)
adata = ad.AnnData(X=df_transposed, obs=df_meta_aligned)

# Save it as a .h5ad file
h5ad_file = "CHOOSE_ready_for_DeepSEM.h5ad"
adata.write_h5ad(h5ad_file)

deepsem_command = [
    "python", "/content/DeepSEM-master/main.py",
    "--task", "celltype_GRN",
    "--data_file", h5ad_file,
    "--save_name", "DeepSEM_global_out",
    "--setting", "test",
    "--n_epochs", "120"
]

# Run DeepSEM
try:
    result = subprocess.run(deepsem_command, capture_output=True, text=True, check=True)
    expected_file = f"DeepSEM_global_out/GRN_inference_result.tsv"
    if os.path.exists(expected_file) and os.path.getsize(expected_file) > 0:
        print(f"Finished processing.")
    else:
        print(f"WARNING: Process did not crash, but no data was found.")
        # Print DeepSEM logs
        print(result.stdout)
    os.remove(h5ad_file)
except subprocess.CalledProcessError as e:
    print(f"Error running DeepSEM.")
    # Print console outputs
    print(e.stdout)
    print(e.stderr)
    if os.path.exists(h5ad_file):
        os.remove(h5ad_file)